In [1]:
from pyspark.sql import SparkSession

# 1. Khởi tạo Spark Session
spark = SparkSession.builder \
    .appName("Colab_Spark_Job") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 21:37:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
identity_path = "./data/test_identity.csv" # Thay đổi đường dẫn tương ứng
identity_df = spark.read.csv(identity_path, header=True, inferSchema=True)

identity_df.show(5)
identity_df.printSchema()

+-------------+-----+--------+-----+-----+-----+-----+-----+-----+-----+-----+-----+--------+-----+------+-----+--------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+--------+-------------+--------------------+-----+--------+--------------+-----+-----+-----+-----+----------+--------------------+
|TransactionID|id-01|   id-02|id-03|id-04|id-05|id-06|id-07|id-08|id-09|id-10|id-11|   id-12|id-13| id-14|id-15|   id-16|id-17|id-18|id-19|id-20|id-21|id-22|id-23|id-24|id-25|id-26|id-27|id-28|   id-29|        id-30|               id-31|id-32|   id-33|         id-34|id-35|id-36|id-37|id-38|DeviceType|          DeviceInfo|
+-------------+-----+--------+-----+-----+-----+-----+-----+-----+-----+-----+-----+--------+-----+------+-----+--------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+--------+-------------+--------------------+-----+--------+--------------+-----+-----+-----+-----+----------+--------------------+
|      3663586|-45.0|280290.

26/04/02 21:37:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [8]:
identity_path = "./data/test_transaction.csv" # Thay đổi đường dẫn tương ứng
transaction_df = spark.read.csv(identity_path, header=True, inferSchema=True)

transaction_df.show(5)
transaction_df.printSchema()

# Lấy số dòng (Action - tốn thời gian tính toán nếu data lớn)
rows = transaction_df.count()

# Lấy số cột (Metadata - lấy cực nhanh)
cols = len(transaction_df.columns)

print(f"Shape: ({rows}, {cols})")

+-------------+-------------+--------------+---------+-----+-----+-----+----------+-----+-----+-----+-----+------+-----+-------------+-------------+---+---+---+---+---+---+---+---+---+---+---+---+-----+---+-----+-----+----+-----+----+----+----+----+----+-----+-----+----+----+----+-----+---+---+---+----+----+---+----+----+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+----+----+----+----+-----+----+----+--

In [4]:
from pyspark.sql import functions as F

# 1. Tính số lượng dòng trong DataFrame
total_rows = transaction_df.count()
threshold = 0.9  # Ngưỡng 90%

# 2. Tính số lượng null cho mỗi cột
null_counts = transaction_df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in transaction_df.columns]).collect()[0].asDict()

# 3. Tìm các cột có tỷ lệ null > 90%
cols_to_drop = [c for c, count in null_counts.items() if (count / total_rows) > threshold]

# 4. Thực hiện drop các cột này
transaction_filtered_df = transaction_df.drop(*cols_to_drop)

# Thông báo kết quả
print(f"Số lượng cột đã xóa: {len(cols_to_drop)}")
print(f"Các cột bị xóa: {cols_to_drop}")

# Hiển thị Schema mới
transaction_filtered_df.printSchema()

Số lượng cột đã xóa: 1
Các cột bị xóa: ['dist2']
root
 |-- TransactionID: integer (nullable = true)
 |-- TransactionDT: integer (nullable = true)
 |-- TransactionAmt: double (nullable = true)
 |-- ProductCD: string (nullable = true)
 |-- card1: integer (nullable = true)
 |-- card2: double (nullable = true)
 |-- card3: double (nullable = true)
 |-- card4: string (nullable = true)
 |-- card5: double (nullable = true)
 |-- card6: string (nullable = true)
 |-- addr1: double (nullable = true)
 |-- addr2: double (nullable = true)
 |-- dist1: double (nullable = true)
 |-- P_emaildomain: string (nullable = true)
 |-- R_emaildomain: string (nullable = true)
 |-- C1: double (nullable = true)
 |-- C2: double (nullable = true)
 |-- C3: double (nullable = true)
 |-- C4: double (nullable = true)
 |-- C5: double (nullable = true)
 |-- C6: double (nullable = true)
 |-- C7: double (nullable = true)
 |-- C8: double (nullable = true)
 |-- C9: double (nullable = true)
 |-- C10: double (nullable = true)
 |